<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/14_NeuroFHIR_Review_WISH_Distinct_Case_Expansion_and_Protocol_Hardening.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/14_NeuroFHIR_Review_WISH_Distinct_Case_Expansion_and_Protocol_Hardening.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeuroFHIR-Review — Notebook 14
## Distinct Case Expansion + Experimental Protocol Hardening for WISH 2026

Notebook 13 successfully created the **research interaction prototype**, but it reused the same three underlying NeuroFHIR-QC demonstration cases across multiple scenarios.

That is acceptable for software prototyping, but it is **not the study design we want for human data collection**, because a participant could recognize the same MRI/trajectory more than once.

This notebook fixes that before recruitment.

### What Notebook 14 does

1. Preserves the frozen NeuroFHIR-QC / AMIA application.
2. Preserves Notebook 13 as the prototype-build record.
3. Selects **12 genuinely distinct public MRI cases** from the same Medical Segmentation Decathlon Task01 BrainTumour source.
4. Excludes the three original NeuroFHIR-QC demonstration source cases.
5. Screens candidate reference masks and chooses a reproducible, volume-diverse 12-case set.
6. Downloads the 12 distinct four-modal MRI volumes and expert reference masks.
7. Runs the **same pinned MONAI BraTS SegResNet** used by Notebook 04.
8. Uses the model's actual current-volume output as the AI-derived measurement.
9. Creates synthetic longitudinal context only for the experimental review task, with that fact explicitly recorded.
10. Assigns each participant **one presentation of each underlying case only**.
11. Keeps 6 Evidence-First + 6 AI-First presentations through counterbalanced sequences.
12. Removes study-answer leakage from the participant-facing app:
    - no `wrong-but-plausible` label,
    - no expected disposition,
    - no reference answer,
    - no source-case ID,
    - no Dice/reference-mask performance,
    - no condition/sequence label shown in the UI.
13. Assigns Sequence A/B **automatically and invisibly** from the pseudonymous participant ID.
14. Keeps the researcher-only scenario key in a separate manifest for later analysis.
15. Produces a hardened pilot-ready WISH app and a final protocol audit.

### Scientific boundary

The selected MRIs are public de-identified research images. The longitudinal prior measurements, dates, AI-confidence displays, QC-state manipulations, and provenance-completeness manipulations used for the WISH experiment are **standardized/synthetic study stimuli**. They do not represent true repeated scans of the public-image donors and do not create clinical ground truth.

For **Path A**, clinical appropriateness/reference interpretation remains pending expert annotation or another legitimate reference procedure.

For **Path B**, the predefined reference outcome is a **workflow disposition rule**, not medical diagnostic accuracy.

> Before collecting real human-participant data, follow the IRB/ethics or exemption process required by the institution conducting the study.


In [1]:
# Cell 1 — Mount Drive, enforce Notebook 13 + source-artifact gates, and create Notebook 14 workspace

from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")
WISH_ROOT = PROJECT_ROOT / "wish_extension"

NB13_AUDIT = (
    WISH_ROOT
    / "evaluation/results/notebook_13_review_foundation/"
    / "notebook_13_neurofhir_review_wish_build_audit.json"
)
NB13_PROTOCOL = WISH_ROOT / "config/research_protocol.json"

NB03_TAR_INDEX = (
    PROJECT_ROOT
    / "evaluation/results/notebook_03_imaging_preparation/"
    / "task01_brain_tumour_tar_index.json"
)
NB03_SELECTION_REPORT = (
    PROJECT_ROOT
    / "evaluation/results/notebook_03_imaging_preparation/"
    / "case_selection_report.json"
)
NB04_MODEL_CHECKPOINT = (
    PROJECT_ROOT
    / "model/brats_mri_segmentation_v0.5.4/models/model.pt"
)

NB14_ROOT = WISH_ROOT / "notebook_14_distinct_case_study"
DATA_ROOT = NB14_ROOT / "data"
IMAGING_ROOT = NB14_ROOT / "imaging"
MASK_ROOT = NB14_ROOT / "masks"
MODEL_OUTPUT_ROOT = NB14_ROOT / "model_outputs"
PREVIEW_ROOT = NB14_ROOT / "previews"
RESEARCH_ROOT = NB14_ROOT / "researcher_only"
PARTICIPANT_ROOT = NB14_ROOT / "participant_app"
DOC_ROOT = NB14_ROOT / "docs"
EVAL_ROOT = NB14_ROOT / "evaluation"
LOCAL_ROOT = Path("/content/neurofhir_wish_nb14")

for folder in (
    DATA_ROOT,
    IMAGING_ROOT,
    MASK_ROOT,
    MODEL_OUTPUT_ROOT,
    PREVIEW_ROOT,
    RESEARCH_ROOT,
    PARTICIPANT_ROOT,
    DOC_ROOT,
    EVAL_ROOT,
    LOCAL_ROOT,
):
    folder.mkdir(parents=True, exist_ok=True)


def utc_now() -> str:
    return (
        datetime.now(timezone.utc)
        .replace(microsecond=0)
        .isoformat()
        .replace("+00:00", "Z")
    )


def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False, allow_nan=False)
        handle.write("\n")
    temporary.replace(path)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


required = [
    NB13_AUDIT,
    NB13_PROTOCOL,
    NB03_TAR_INDEX,
    NB03_SELECTION_REPORT,
]
missing = [str(p) for p in required if not p.exists() or p.stat().st_size == 0]
if missing:
    raise FileNotFoundError(
        "Required prior artifacts are missing:\n"
        + "\n".join(f" - {p}" for p in missing)
    )

nb13_audit = load_json(NB13_AUDIT)
if nb13_audit.get("status") != "completed" or not nb13_audit.get("final_gate"):
    raise RuntimeError("Notebook 13 did not pass its final gate.")

protocol = load_json(NB13_PROTOCOL)
tar_index = load_json(NB03_TAR_INDEX)
nb03_selection = load_json(NB03_SELECTION_REPORT)

original_source_cases = {
    row["source_case_id"]
    for row in nb03_selection.get("selected_cases", [])
    if row.get("source_case_id")
}

print("=" * 108)
print("✅ Notebook 13 gate passed")
print(f"✅ Cached MSD TAR index loaded: {tar_index.get('member_count')} members")
print(f"✅ Original demonstration source cases excluded: {sorted(original_source_cases)}")
print(f"✅ Notebook 14 isolated workspace: {NB14_ROOT}")
print("✅ Existing NeuroFHIR-QC / AMIA and Notebook 13 artifacts will not be overwritten")
print("=" * 108)


Mounted at /content/drive
✅ Notebook 13 gate passed
✅ Cached MSD TAR index loaded: 1274 members
✅ Original demonstration source cases excluded: ['BRATS_029', 'BRATS_167', 'BRATS_443']
✅ Notebook 14 isolated workspace: /content/drive/MyDrive/neurofhir-qc/wish_extension/notebook_14_distinct_case_study
✅ Existing NeuroFHIR-QC / AMIA and Notebook 13 artifacts will not be overwritten


In [2]:
# Cell 2 — Install runtime, connect to the public MSD TAR archive, and select 12 distinct source cases

required_packages = [
    "boto3>=1.34,<2",
    "nibabel>=5.2,<6",
    "monai==1.6.0",
    "huggingface_hub>=0.30,<2",
    "scipy>=1.11,<2",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", *required_packages]
)

import boto3
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import scipy
import torch
from botocore import UNSIGNED
from botocore.client import Config as BotoConfig
from huggingface_hub import snapshot_download
from monai.inferers import sliding_window_inference
from monai.networks.nets import SegResNet

AWS_BUCKET = tar_index["archive_bucket"]
AWS_REGION = tar_index.get("archive_region", "us-west-2")
ARCHIVE_KEY = tar_index["archive_key"]
ARCHIVE_ETAG = tar_index["archive_etag"]
ARCHIVE_SIZE = int(tar_index["archive_size_bytes"])

s3 = boto3.client(
    "s3",
    region_name=AWS_REGION,
    config=BotoConfig(
        signature_version=UNSIGNED,
        retries={"max_attempts": 5, "mode": "standard"},
    ),
)

head = s3.head_object(Bucket=AWS_BUCKET, Key=ARCHIVE_KEY)
if int(head["ContentLength"]) != ARCHIVE_SIZE:
    raise RuntimeError("MSD TAR size no longer matches the cached Notebook 03 index.")

current_etag = str(head.get("ETag", "")).strip('"')
if current_etag != ARCHIVE_ETAG:
    raise RuntimeError("MSD TAR ETag changed; regenerate the Notebook 03 TAR index first.")

members = tar_index["members"]


def norm_member(name: str) -> str:
    name = name.replace("\\", "/")
    while name.startswith("./"):
        name = name[2:]
    return name.lstrip("/")


member_by_name = {norm_member(m["name"]): m for m in members}

image_keys = sorted(
    name
    for name in member_by_name
    if "/imagesTr/" in name and name.endswith(".nii.gz")
)
label_keys = sorted(
    name
    for name in member_by_name
    if "/labelsTr/" in name and name.endswith(".nii.gz")
)


def nifti_stem(key: str) -> str:
    name = Path(key).name
    return name[:-7] if name.endswith(".nii.gz") else Path(name).stem


images_by_stem = {nifti_stem(k): k for k in image_keys}
labels_by_stem = {nifti_stem(k): k for k in label_keys}


def archive_metadata_key(key: str) -> bool:
    parts = [x for x in norm_member(key).split("/") if x]
    return any(part == "__MACOSX" or part.startswith("._") for part in parts)


eligible = sorted(
    stem
    for stem in set(images_by_stem) & set(labels_by_stem)
    if (
        not stem.startswith("._")
        and not archive_metadata_key(images_by_stem[stem])
        and not archive_metadata_key(labels_by_stem[stem])
        and stem not in original_source_cases
    )
)

if len(eligible) < 12:
    raise RuntimeError(f"Only {len(eligible)} eligible distinct source cases remain.")


def download_member(member_name: str, destination: Path) -> dict[str, Any]:
    member_name = norm_member(member_name)
    member = member_by_name.get(member_name)
    if member is None:
        raise FileNotFoundError(member_name)

    expected = int(member["size_bytes"])
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists() and destination.stat().st_size == expected:
        return {
            "member": member_name,
            "size_bytes": expected,
            "cached": True,
            "sha256": sha256_file(destination),
        }

    tmp = destination.with_suffix(destination.suffix + ".part")
    if tmp.exists():
        tmp.unlink()

    response = s3.get_object(
        Bucket=AWS_BUCKET,
        Key=ARCHIVE_KEY,
        Range=f"bytes={int(member['data_offset'])}-{int(member['data_end_offset'])}",
    )
    with tmp.open("wb") as handle:
        while True:
            chunk = response["Body"].read(1024 * 1024)
            if not chunk:
                break
            handle.write(chunk)

    if tmp.stat().st_size != expected:
        raise IOError(
            f"Size mismatch for {member_name}: {tmp.stat().st_size} != {expected}"
        )

    tmp.replace(destination)
    return {
        "member": member_name,
        "size_bytes": expected,
        "cached": False,
        "sha256": sha256_file(destination),
    }


def reference_volume_ml(mask_path: Path) -> float:
    image = nib.load(str(mask_path))
    arr = np.rint(np.asanyarray(image.dataobj)).astype(np.int16)
    if arr.ndim != 3:
        raise ValueError(f"Expected a 3D label mask, found {arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"Non-finite mask values: {mask_path}")
    voxels = int(np.count_nonzero(arr > 0))
    if voxels <= 0:
        raise ValueError(f"Empty tumor mask: {mask_path}")
    spacing = tuple(float(v) for v in image.header.get_zooms()[:3])
    return float(voxels * np.prod(spacing) / 1000.0)


# Screen an evenly spaced 48-case set, then choose 12 that cover the reference-volume
# distribution. This is for stimulus diversity, NOT a performance sample.
screen_count = min(48, len(eligible))
screen_indices = sorted(
    {
        int(round(x))
        for x in np.linspace(0, len(eligible) - 1, num=screen_count)
    }
)
screen_stems = [eligible[i] for i in screen_indices]

screen_root = LOCAL_ROOT / "screen_labels"
screen_root.mkdir(parents=True, exist_ok=True)

screen_rows = []
for i, stem in enumerate(screen_stems, start=1):
    local_label = screen_root / f"{stem}.nii.gz"
    download_member(labels_by_stem[stem], local_label)
    volume = reference_volume_ml(local_label)
    screen_rows.append({"source_case_id": stem, "reference_volume_ml": volume})
    print(f"[{i:02d}/{len(screen_stems):02d}] {stem}: {volume:.2f} mL")

screen_df = pd.DataFrame(screen_rows).sort_values("reference_volume_ml").reset_index(drop=True)

# Choose nearest unique cases to 12 evenly spaced quantile positions.
target_positions = np.linspace(0, len(screen_df) - 1, 12)
chosen_indices = []
for x in target_positions:
    ranked = sorted(
        range(len(screen_df)),
        key=lambda idx: (abs(idx - x), idx),
    )
    chosen = next(idx for idx in ranked if idx not in chosen_indices)
    chosen_indices.append(chosen)

selected_df = screen_df.iloc[chosen_indices].copy().sort_values("reference_volume_ml")
selected_df["participant_case_id"] = [
    f"WISH-{i:02d}" for i in range(1, len(selected_df) + 1)
]

if selected_df["source_case_id"].nunique() != 12:
    raise AssertionError("Distinct-case selection failed.")

SELECTION_CSV = DATA_ROOT / "distinct_case_selection.csv"
SELECTION_JSON = DATA_ROOT / "distinct_case_selection.json"
selected_df.to_csv(SELECTION_CSV, index=False)

selection_payload = {
    "generated_utc": utc_now(),
    "selection_method": (
        "Screen 48 evenly spaced eligible MSD Task01 label masks after excluding "
        "the three original NeuroFHIR-QC demonstration source cases, then choose "
        "12 unique cases spanning the screened reference-volume distribution."
    ),
    "selection_purpose": "stimulus diversity; not model-performance sampling",
    "excluded_original_source_cases": sorted(original_source_cases),
    "screened_count": len(screen_df),
    "selected_count": len(selected_df),
    "selected_cases": selected_df.to_dict(orient="records"),
}
write_json(SELECTION_JSON, selection_payload)

print("=" * 108)
print("✅ 12 distinct public MRI source cases selected")
print(f"✅ Original 3 source cases excluded: {selected_df['source_case_id'].isin(original_source_cases).sum() == 0}")
print(f"✅ Reference-volume range: {selected_df['reference_volume_ml'].min():.2f}–{selected_df['reference_volume_ml'].max():.2f} mL")
print(f"📄 {SELECTION_JSON}")
print("=" * 108)


[01/48] BRATS_001: 111.72 mL
[02/48] BRATS_011: 137.30 mL
[03/48] BRATS_021: 49.64 mL
[04/48] BRATS_033: 114.02 mL
[05/48] BRATS_043: 40.76 mL
[06/48] BRATS_053: 74.29 mL
[07/48] BRATS_063: 143.78 mL
[08/48] BRATS_073: 109.92 mL
[09/48] BRATS_084: 28.39 mL
[10/48] BRATS_094: 164.98 mL
[11/48] BRATS_104: 46.77 mL
[12/48] BRATS_114: 72.61 mL
[13/48] BRATS_125: 127.23 mL
[14/48] BRATS_135: 171.42 mL
[15/48] BRATS_145: 102.06 mL
[16/48] BRATS_155: 263.75 mL
[17/48] BRATS_165: 164.46 mL
[18/48] BRATS_177: 12.85 mL
[19/48] BRATS_187: 39.83 mL
[20/48] BRATS_197: 113.29 mL
[21/48] BRATS_207: 201.30 mL
[22/48] BRATS_217: 218.84 mL
[23/48] BRATS_228: 117.09 mL
[24/48] BRATS_238: 16.33 mL
[25/48] BRATS_248: 61.30 mL
[26/48] BRATS_258: 55.49 mL
[27/48] BRATS_269: 173.04 mL
[28/48] BRATS_279: 187.57 mL
[29/48] BRATS_289: 82.73 mL
[30/48] BRATS_299: 21.96 mL
[31/48] BRATS_309: 125.29 mL
[32/48] BRATS_320: 180.62 mL
[33/48] BRATS_330: 14.83 mL
[34/48] BRATS_340: 36.34 mL
[35/48] BRATS_350: 55.36 mL
[

In [3]:
# Cell 3 — Download and standardize the 12 four-modal MRI cases + reference masks + neutral previews

prepared_cases = []

# MSD Task01 channel order from the source dataset:
# 0=FLAIR, 1=T1w, 2=T1gd/T1c, 3=T2w
source_channel_map = {
    "FLAIR": 0,
    "T1": 1,
    "T1c": 2,
    "T2": 3,
}
model_input_order = ["T1c", "T1", "T2", "FLAIR"]

for row in selected_df.to_dict(orient="records"):
    pid = row["participant_case_id"]
    stem = row["source_case_id"]

    case_image_root = IMAGING_ROOT / pid
    case_mask_root = MASK_ROOT / pid
    case_image_root.mkdir(parents=True, exist_ok=True)
    case_mask_root.mkdir(parents=True, exist_ok=True)

    raw_image = LOCAL_ROOT / "raw_images" / f"{stem}.nii.gz"
    raw_label = LOCAL_ROOT / "raw_labels" / f"{stem}.nii.gz"

    image_transfer = download_member(images_by_stem[stem], raw_image)
    label_transfer = download_member(labels_by_stem[stem], raw_label)

    image_4d = nib.load(str(raw_image))
    image_arr = np.asanyarray(image_4d.dataobj)

    if image_arr.ndim != 4 or image_arr.shape[-1] != 4:
        raise AssertionError(f"{stem}: expected [X,Y,Z,4], found {image_arr.shape}")
    if not np.isfinite(image_arr).all():
        raise ValueError(f"{stem}: non-finite MRI values")

    reference_img = nib.load(str(raw_label))
    reference_arr = np.rint(np.asanyarray(reference_img.dataobj)).astype(np.uint8)

    if tuple(reference_arr.shape) != tuple(image_arr.shape[:3]):
        raise AssertionError(
            f"{stem}: MRI/reference shape mismatch {image_arr.shape[:3]} vs {reference_arr.shape}"
        )
    if not np.allclose(image_4d.affine, reference_img.affine, atol=1e-4):
        raise AssertionError(f"{stem}: MRI/reference affine mismatch")

    modality_paths = {}
    for modality, source_index in source_channel_map.items():
        output = case_image_root / f"{pid}_{modality}.nii.gz"
        header = image_4d.header.copy()
        header.set_data_dtype(np.float32)
        nib.save(
            nib.Nifti1Image(
                image_arr[..., source_index].astype(np.float32),
                image_4d.affine,
                header,
            ),
            str(output),
        )
        modality_paths[modality] = str(output)

    ref_multiclass = case_mask_root / "reference_multiclass.nii.gz"
    ref_whole = case_mask_root / "reference_whole_tumor.nii.gz"

    ref_header = reference_img.header.copy()
    ref_header.set_data_dtype(np.uint8)
    nib.save(
        nib.Nifti1Image(reference_arr, reference_img.affine, ref_header),
        str(ref_multiclass),
    )
    nib.save(
        nib.Nifti1Image(
            (reference_arr > 0).astype(np.uint8),
            reference_img.affine,
            ref_header,
        ),
        str(ref_whole),
    )

    # Neutral preview: T1c with reference contour, no condition or answer text.
    lesion = reference_arr > 0
    z_scores = lesion.sum(axis=(0, 1))
    slice_index = int(np.argmax(z_scores))
    base = image_arr[:, :, slice_index, source_channel_map["T1c"]]

    fig = plt.figure(figsize=(5, 5))
    plt.imshow(np.rot90(base), cmap="gray")
    if lesion[:, :, slice_index].any():
        plt.contour(
            np.rot90(lesion[:, :, slice_index]),
            levels=[0.5],
            linewidths=1.0,
        )
    plt.title(pid)
    plt.axis("off")
    preview_path = PREVIEW_ROOT / f"{pid}.png"
    plt.tight_layout()
    fig.savefig(preview_path, dpi=160, bbox_inches="tight")
    plt.close(fig)

    prepared_cases.append(
        {
            "participant_case_id": pid,
            "source_case_id": stem,
            "source_reference_volume_ml": float(row["reference_volume_ml"]),
            "modality_paths": modality_paths,
            "model_input_order": model_input_order,
            "reference_multiclass_path": str(ref_multiclass),
            "reference_whole_tumor_path": str(ref_whole),
            "preview_path": str(preview_path),
            "image_transfer": image_transfer,
            "label_transfer": label_transfer,
        }
    )

PREPARED_JSON = DATA_ROOT / "prepared_distinct_cases.json"
write_json(
    PREPARED_JSON,
    {
        "generated_utc": utc_now(),
        "case_count": len(prepared_cases),
        "cases": prepared_cases,
        "boundary": (
            "Public de-identified MRI; no claim that synthetic longitudinal context "
            "represents true repeated imaging of the source donor."
        ),
    },
)

assert len(prepared_cases) == 12
assert len({c["source_case_id"] for c in prepared_cases}) == 12

print("=" * 108)
print("✅ 12 distinct MRI cases standardized")
print("✅ 48 modality files created")
print("✅ 12 expert reference masks preserved")
print("✅ 12 neutral participant previews created")
print(f"📄 {PREPARED_JSON}")
print("=" * 108)


✅ 12 distinct MRI cases standardized
✅ 48 modality files created
✅ 12 expert reference masks preserved
✅ 12 neutral participant previews created
📄 /content/drive/MyDrive/neurofhir-qc/wish_extension/notebook_14_distinct_case_study/data/prepared_distinct_cases.json


In [4]:
# Cell 4 — Run the same pinned MONAI BraTS model on all 12 distinct cases

if not torch.cuda.is_available():
    raise RuntimeError(
        "Notebook 14 requires a GPU for the 12-case inference stage. "
        "In Colab choose Runtime → Change runtime type → GPU and rerun."
    )

DEVICE = torch.device("cuda:0")
GPU = torch.cuda.get_device_properties(0)
GPU_MEMORY_GB = GPU.total_memory / (1024 ** 3)

if GPU_MEMORY_GB >= 32:
    ROI_SIZE = (240, 240, 160)
elif GPU_MEMORY_GB >= 20:
    ROI_SIZE = (192, 192, 144)
else:
    ROI_SIZE = (160, 160, 128)

SW_BATCH_SIZE = 1
SW_OVERLAP = 0.5
MODEL_THRESHOLD = 0.5

MODEL_BUNDLE_NAME = "brats_mri_segmentation"
MODEL_BUNDLE_VERSION = "0.5.4"
MODEL_BUNDLE_REVISION = "370f7f9d062745fbac445e7fe6d6616d35df04ec"
MODEL_REPO_ID = "MONAI/brats_mri_segmentation"
MODEL_EXPECTED_SHA256 = (
    "860ccb3f1c21c99d0410ad8a1ac4ef6b8fab60cec0a503b0ba42675741a750ae"
)
MODEL_ROOT = PROJECT_ROOT / f"model/{MODEL_BUNDLE_NAME}_v{MODEL_BUNDLE_VERSION}"
MODEL_CHECKPOINT = MODEL_ROOT / "models/model.pt"

if not (
    MODEL_CHECKPOINT.exists()
    and MODEL_CHECKPOINT.stat().st_size > 1024 * 1024
    and sha256_file(MODEL_CHECKPOINT) == MODEL_EXPECTED_SHA256
):
    MODEL_ROOT.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=MODEL_REPO_ID,
        revision=MODEL_BUNDLE_REVISION,
        local_dir=str(MODEL_ROOT),
        allow_patterns=[
            "models/model.pt",
            "configs/metadata.json",
            "configs/inference.json",
            "docs/README.md",
            "LICENSE",
        ],
        max_workers=4,
    )

if sha256_file(MODEL_CHECKPOINT) != MODEL_EXPECTED_SHA256:
    raise RuntimeError("Pinned model checkpoint SHA-256 verification failed.")


def extract_state_dict(checkpoint: Any) -> dict[str, torch.Tensor]:
    if isinstance(checkpoint, dict):
        for key in ("model", "state_dict", "network"):
            candidate = checkpoint.get(key)
            if isinstance(candidate, dict) and candidate:
                return candidate
        if checkpoint and all(torch.is_tensor(v) for v in checkpoint.values()):
            return checkpoint
    raise TypeError("Could not identify model state dictionary.")


def clean_state_dict(state: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    cleaned = {}
    for key, value in state.items():
        clean_key = key
        for prefix in ("module.", "_orig_mod."):
            if clean_key.startswith(prefix):
                clean_key = clean_key[len(prefix):]
        cleaned[clean_key] = value
    return cleaned


network = SegResNet(
    spatial_dims=3,
    init_filters=16,
    in_channels=4,
    out_channels=3,
    dropout_prob=0.2,
    blocks_down=(1, 2, 2, 4),
    blocks_up=(1, 1, 1),
)

try:
    checkpoint = torch.load(MODEL_CHECKPOINT, map_location="cpu", weights_only=True)
except Exception:
    checkpoint = torch.load(MODEL_CHECKPOINT, map_location="cpu", weights_only=False)

load_result = network.load_state_dict(
    clean_state_dict(extract_state_dict(checkpoint)),
    strict=True,
)
if load_result.missing_keys or load_result.unexpected_keys:
    raise RuntimeError("Pinned checkpoint did not strictly match the expected SegResNet.")

network = network.to(DEVICE)
network.eval()
del checkpoint


def normalize_nonzero_channels(channels: np.ndarray) -> np.ndarray:
    if channels.ndim != 4 or channels.shape[0] != 4:
        raise ValueError(f"Expected [4,X,Y,Z], got {channels.shape}")
    result = np.zeros_like(channels, dtype=np.float32)
    for i in range(4):
        channel = channels[i].astype(np.float32, copy=False)
        mask = np.isfinite(channel) & (channel != 0)
        if not mask.any():
            raise ValueError(f"Input channel {i} has no nonzero finite voxels.")
        vals = channel[mask]
        std = float(vals.std())
        if std <= 1e-8:
            raise ValueError(f"Input channel {i} has near-zero SD.")
        result[i, mask] = (vals - float(vals.mean())) / std
    return result


def run_model(channels: np.ndarray) -> tuple[np.ndarray, float]:
    normalized = normalize_nonzero_channels(channels)
    tensor = torch.from_numpy(normalized[None]).to(DEVICE, non_blocking=True)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True):
            logits = sliding_window_inference(
                tensor,
                roi_size=ROI_SIZE,
                sw_batch_size=SW_BATCH_SIZE,
                predictor=network,
                overlap=SW_OVERLAP,
                mode="gaussian",
            )
        probabilities = torch.sigmoid(logits).float()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    out = probabilities[0].cpu().numpy().astype(np.float32, copy=False)
    del tensor, logits, probabilities
    torch.cuda.empty_cache()
    return out, elapsed


def nested_prediction(probabilities: np.ndarray) -> dict[str, np.ndarray]:
    tc_raw = probabilities[0] >= MODEL_THRESHOLD
    wt_raw = probabilities[1] >= MODEL_THRESHOLD
    et_raw = probabilities[2] >= MODEL_THRESHOLD

    et = et_raw
    tc = tc_raw | et
    wt = wt_raw | tc

    multiclass = np.zeros(wt.shape, dtype=np.uint8)
    multiclass[wt] = 2
    multiclass[tc] = 1
    multiclass[et] = 3

    return {
        "tumor_core": tc.astype(np.uint8),
        "whole_tumor": wt.astype(np.uint8),
        "enhancing_tumor": et.astype(np.uint8),
        "multiclass": multiclass,
    }


def dice_score(pred: np.ndarray, ref: np.ndarray) -> float:
    p = pred.astype(bool)
    r = ref.astype(bool)
    tp = int(np.count_nonzero(p & r))
    denom = int(np.count_nonzero(p)) + int(np.count_nonzero(r))
    return 1.0 if denom == 0 else (2.0 * tp) / denom


def volume_ml(mask: np.ndarray, spacing: tuple[float, float, float]) -> float:
    return float(np.count_nonzero(mask) * np.prod(spacing) / 1000.0)


torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

model_rows = []

for i, case in enumerate(prepared_cases, start=1):
    pid = case["participant_case_id"]
    print(f"[{i:02d}/12] Running {pid} ...")

    arrays = []
    for modality in model_input_order:
        img = nib.load(case["modality_paths"][modality])
        arr = np.asanyarray(img.dataobj).astype(np.float32, copy=False)
        arrays.append(arr)
    channels = np.stack(arrays, axis=0)

    ref_img = nib.load(case["reference_multiclass_path"])
    ref = np.rint(np.asanyarray(ref_img.dataobj)).astype(np.uint8)
    spacing = tuple(float(v) for v in ref_img.header.get_zooms()[:3])

    probabilities, elapsed = run_model(channels)
    prediction = nested_prediction(probabilities)

    if probabilities.shape[1:] != ref.shape:
        raise AssertionError(f"{pid}: model/reference shape mismatch")
    if not prediction["whole_tumor"].any():
        raise AssertionError(f"{pid}: empty whole-tumor prediction")

    pred_volume = volume_ml(prediction["whole_tumor"], spacing)
    ref_volume = volume_ml(ref > 0, spacing)
    wt_dice = dice_score(prediction["whole_tumor"], ref > 0)

    output_root = MODEL_OUTPUT_ROOT / pid
    output_root.mkdir(parents=True, exist_ok=True)

    pred_path = output_root / "predicted_whole_tumor.nii.gz"
    header = ref_img.header.copy()
    header.set_data_dtype(np.uint8)
    nib.save(
        nib.Nifti1Image(
            prediction["whole_tumor"],
            ref_img.affine,
            header,
        ),
        str(pred_path),
    )

    model_rows.append(
        {
            "participant_case_id": pid,
            "source_case_id": case["source_case_id"],
            "predicted_current_volume_ml": round(pred_volume, 6),
            "source_reference_volume_ml": round(ref_volume, 6),
            "whole_tumor_dice": round(float(wt_dice), 6),
            "inference_seconds": round(float(elapsed), 6),
            "prediction_path": str(pred_path),
        }
    )
    print(
        f"    predicted={pred_volume:.2f} mL | "
        f"reference={ref_volume:.2f} mL | "
        f"Dice={wt_dice:.4f} | {elapsed:.2f}s"
    )

MODEL_RESULTS_CSV = DATA_ROOT / "distinct_case_model_results.csv"
MODEL_RESULTS_JSON = DATA_ROOT / "distinct_case_model_results.json"
pd.DataFrame(model_rows).to_csv(MODEL_RESULTS_CSV, index=False)
write_json(
    MODEL_RESULTS_JSON,
    {
        "generated_utc": utc_now(),
        "model": {
            "bundle": MODEL_BUNDLE_NAME,
            "version": MODEL_BUNDLE_VERSION,
            "revision": MODEL_BUNDLE_REVISION,
            "checkpoint_sha256": MODEL_EXPECTED_SHA256,
            "architecture": "MONAI SegResNet",
            "input_order": model_input_order,
        },
        "interpretation_boundary": (
            "These are demonstration model outputs on public MSD cases. "
            "They are not independent external or clinical validation."
        ),
        "cases": model_rows,
    },
)

print("=" * 108)
print("✅ Same pinned MONAI model executed on 12 distinct cases")
print(f"✅ Mean WT Dice: {pd.DataFrame(model_rows)['whole_tumor_dice'].mean():.4f}")
print(f"✅ Mean inference: {pd.DataFrame(model_rows)['inference_seconds'].mean():.2f}s")
print("⚠️ Performance metrics are researcher-only and will NOT appear in the participant app")
print("=" * 108)


[01/12] Running WISH-01 ...
    predicted=24.03 mL | reference=12.85 mL | Dice=0.5702 | 2.47s
[02/12] Running WISH-02 ...
    predicted=22.32 mL | reference=26.81 mL | Dice=0.8502 | 1.19s
[03/12] Running WISH-03 ...
    predicted=47.88 mL | reference=44.80 mL | Dice=0.9447 | 1.12s
[04/12] Running WISH-04 ...
    predicted=45.84 mL | reference=55.36 mL | Dice=0.8466 | 1.14s
[05/12] Running WISH-05 ...
    predicted=67.92 mL | reference=72.61 mL | Dice=0.9109 | 1.14s
[06/12] Running WISH-06 ...
    predicted=89.92 mL | reference=91.56 mL | Dice=0.9659 | 1.13s
[07/12] Running WISH-07 ...
    predicted=125.08 mL | reference=111.72 mL | Dice=0.9130 | 1.13s
[08/12] Running WISH-08 ...
    predicted=119.78 mL | reference=125.29 mL | Dice=0.9423 | 1.13s
[09/12] Running WISH-09 ...
    predicted=131.06 mL | reference=143.78 mL | Dice=0.9343 | 1.13s
[10/12] Running WISH-10 ...
    predicted=156.34 mL | reference=164.98 mL | Dice=0.9319 | 1.14s
[11/12] Running WISH-11 ...
    predicted=171.17 mL 

In [5]:
# Cell 5 — Create 12 one-case-per-participant WISH stimuli and separate researcher-only vs participant-safe data

model_by_pid = {row["participant_case_id"]: row for row in model_rows}
prepared_by_pid = {row["participant_case_id"]: row for row in prepared_cases}

# Experimental matrix. Each row is assigned to one unique underlying MRI case.
# The participant never sees these research labels.
design = [
    ("concordant-high", "stable", "correct", "high", "pass", "complete", "accept"),
    ("concordant-high", "progression", "correct", "high", "pass", "complete", "accept"),
    ("correct-uncertain", "stable", "correct", "low", "borderline", "complete", "flag"),
    ("correct-uncertain", "progression", "correct", "low", "borderline", "complete", "flag"),
    ("wrong-but-plausible", "stable", "wrong", "high", "pass", "complete", "escalate"),
    ("wrong-but-plausible", "progression", "wrong", "high", "pass", "complete", "escalate"),
    ("provenance-incomplete", "stable", "correct", "high", "pass", "incomplete", "flag"),
    ("provenance-incomplete", "progression", "correct", "high", "pass", "incomplete", "flag"),
    ("qc-failure", "stable", "correct", "high", "fail", "complete", "escalate"),
    ("qc-failure", "progression", "correct", "high", "fail", "complete", "escalate"),
    ("longitudinal-discordance", "stable", "wrong", "high", "pass", "complete", "escalate"),
    ("longitudinal-discordance", "progression", "wrong", "high", "pass", "complete", "escalate"),
]

selected_pids = sorted(model_by_pid)
if len(selected_pids) != 12:
    raise AssertionError("Expected exactly 12 model-executed distinct participant cases.")


def make_synthetic_longitudinal(current_volume: float, trajectory: str) -> dict[str, float | str]:
    # Synthetic prior values are constructed from the actual model-derived current volume.
    # Stable target = +3%; progression target = +60%.
    if trajectory == "stable":
        target_change = 3.0
    elif trajectory == "progression":
        target_change = 60.0
    else:
        raise ValueError(trajectory)

    prior = current_volume / (1.0 + target_change / 100.0)
    observed_change = ((current_volume - prior) / prior) * 100.0
    return {
        "trajectory_label_for_researcher": trajectory,
        "prior_volume_ml": round(prior, 3),
        "current_volume_ml": round(current_volume, 3),
        "percent_change": round(observed_change, 2),
        "context_type": "synthetic-standardized-longitudinal-context",
    }


researcher_cases = []
participant_cases = []

for idx, (pid, row_design) in enumerate(zip(selected_pids, design), start=1):
    (
        case_type,
        trajectory,
        ai_correctness,
        confidence_label,
        displayed_qc,
        provenance_state,
        reference_workflow_disposition,
    ) = row_design

    model_row = model_by_pid[pid]
    source_row = prepared_by_pid[pid]
    longitudinal = make_synthetic_longitudinal(
        float(model_row["predicted_current_volume_ml"]),
        trajectory,
    )

    correct_ai_conclusion = "Stable" if trajectory == "stable" else "Progression"
    ai_conclusion = correct_ai_conclusion
    if ai_correctness == "wrong":
        ai_conclusion = "Progression" if correct_ai_conclusion == "Stable" else "Stable"

    displayed_confidence_score = (
        0.91 if confidence_label == "high" else 0.39
    )

    scenario_id = f"S{idx:02d}"

    researcher_case = {
        "scenario_id": scenario_id,
        "participant_case_id": pid,
        "source_case_id": source_row["source_case_id"],
        "case_type": case_type,
        "trajectory": trajectory,
        "ai_correctness": ai_correctness,
        "reference_workflow_disposition_path_b": reference_workflow_disposition,
        "path_A_reference_status": "pending-legitimate-expert-reference",
        "actual_model_metrics_researcher_only": {
            "whole_tumor_dice": model_row["whole_tumor_dice"],
            "source_reference_volume_ml": model_row["source_reference_volume_ml"],
            "predicted_current_volume_ml": model_row["predicted_current_volume_ml"],
        },
        "stimulus": {
            "longitudinal": longitudinal,
            "ai_conclusion": ai_conclusion,
            "ai_confidence_label": confidence_label,
            "ai_confidence_display_score": displayed_confidence_score,
            "displayed_qc_state": displayed_qc,
            "provenance_state": provenance_state,
        },
        "condition_by_sequence": {
            "A": "evidence-first" if idx % 2 == 1 else "ai-first",
            "B": "ai-first" if idx % 2 == 1 else "evidence-first",
        },
    }
    researcher_cases.append(researcher_case)

    # Participant-facing record intentionally excludes:
    # source_case_id, case_type, correctness, expected workflow disposition,
    # reference volume, Dice, and researcher trajectory label.
    participant_case = {
        "scenario_id": scenario_id,
        "case_id": pid,
        "display_title": f"Review Case {idx:02d}",
        "preview_file": f"assets/{pid}.png",
        "longitudinal": {
            "prior_volume_ml": longitudinal["prior_volume_ml"],
            "current_volume_ml": longitudinal["current_volume_ml"],
            "percent_change": longitudinal["percent_change"],
            "context_note": (
                "Standardized longitudinal research context; not a true repeated-scan "
                "history of the public-image donor."
            ),
        },
        "ai": {
            "conclusion": ai_conclusion,
            "confidence_label": confidence_label,
            "confidence_display_score": displayed_confidence_score,
            "displayed_qc_state": displayed_qc,
            "known_limitation": (
                "AI confidence is a study/workflow display signal and is not a "
                "calibrated clinical probability."
            ),
        },
        "passport": {
            "model_name": "MONAI brats_mri_segmentation",
            "model_version": MODEL_BUNDLE_VERSION,
            "intended_use": "Research workflow support; not diagnostic use.",
            "input_qc": displayed_qc,
            "provenance_state": provenance_state,
            "provenance_note": (
                "Lineage available for inspection."
                if provenance_state == "complete"
                else "Some lineage fields are unavailable in this standardized study case."
            ),
            "processing_context": (
                "AI-derived volumetry generated with the pinned NeuroFHIR-QC model; "
                "longitudinal context is standardized for this formative study."
            ),
        },
    }
    participant_cases.append(participant_case)

RESEARCH_MANIFEST = RESEARCH_ROOT / "researcher_scenario_key.json"
PARTICIPANT_MANIFEST = PARTICIPANT_ROOT / "participant_cases.json"

write_json(
    RESEARCH_MANIFEST,
    {
        "generated_utc": utc_now(),
        "case_count": 12,
        "cases": researcher_cases,
        "analysis_join_key": "scenario_id",
        "warning": "Researcher-only. Do not give this file to participants.",
    },
)

write_json(
    PARTICIPANT_MANIFEST,
    {
        "generated_utc": utc_now(),
        "project": "NeuroFHIR-Review",
        "case_count": 12,
        "cases": participant_cases,
        "path_A_initial_labels": ["Stable", "Progression", "Uncertain"],
        "path_B_initial_labels": [
            "Evidence sufficient",
            "Flag concern",
            "Escalate for expert review",
        ],
        "final_actions": [
            "Accept AI",
            "Keep initial judgment",
            "Amend",
            "Reject AI",
            "Escalate",
        ],
        "reason_codes": [
            "evidence-supports-ai",
            "ai-conflicts-with-longitudinal-evidence",
            "low-ai-confidence",
            "qc-failure",
            "missing-or-incomplete-provenance",
            "model-limitation-relevant",
            "evidence-incomplete",
            "uncertain-requires-expert-review",
            "other",
        ],
        "study_boundary": (
            "Formative research prototype. Public de-identified MRI plus synthetic/"
            "standardized longitudinal workflow context. Not patient-care use."
        ),
    },
)

# Copy neutral preview assets into the participant app package.
asset_root = PARTICIPANT_ROOT / "assets"
if asset_root.exists():
    shutil.rmtree(asset_root)
asset_root.mkdir(parents=True, exist_ok=True)
for case in participant_cases:
    src = PREVIEW_ROOT / f"{case['case_id']}.png"
    shutil.copy2(src, asset_root / src.name)

print("=" * 108)
print("✅ 12 unique MRI cases → 12 unique experimental scenarios")
print("✅ Each underlying MRI appears only once per participant")
print("✅ Research answers/metrics separated from participant-facing JSON")
print(f"🔒 Researcher key: {RESEARCH_MANIFEST}")
print(f"👤 Participant-safe data: {PARTICIPANT_MANIFEST}")
print("=" * 108)


✅ 12 unique MRI cases → 12 unique experimental scenarios
✅ Each underlying MRI appears only once per participant
✅ Research answers/metrics separated from participant-facing JSON
🔒 Researcher key: /content/drive/MyDrive/neurofhir-qc/wish_extension/notebook_14_distinct_case_study/researcher_only/researcher_scenario_key.json
👤 Participant-safe data: /content/drive/MyDrive/neurofhir-qc/wish_extension/notebook_14_distinct_case_study/participant_app/participant_cases.json


In [6]:
# Cell 6 — Build the hardened participant app with hidden automatic counterbalancing

APP_HTML = PARTICIPANT_ROOT / "index.html"

html = r"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<meta name="robots" content="noindex">
<title>NeuroFHIR-Review Study</title>
<style>
:root{font-family:Inter,system-ui,-apple-system,Segoe UI,sans-serif;color:#15242d;background:#f4f7f9}
*{box-sizing:border-box}body{margin:0}.shell{max-width:1040px;margin:auto;padding:28px 18px 60px}
header{display:flex;justify-content:space-between;gap:20px;align-items:flex-start;margin-bottom:18px}
h1{margin:5px 0;font-size:38px}h2,h3{margin-top:0}.eyebrow{text-transform:uppercase;letter-spacing:.08em;font-size:12px;font-weight:800;color:#0f766e}
.card{background:#fff;border:1px solid #d9e3e8;border-radius:16px;padding:22px;margin:15px 0;box-shadow:0 8px 24px rgba(20,50,65,.06)}
.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(200px,1fr));gap:12px}
.metric{padding:14px;border:1px solid #e0e8ec;border-radius:10px;background:#f8fafb}
.metric span{display:block;color:#697b86;font-size:12px}.metric b{font-size:20px}.muted{color:#60717c}
label{display:grid;gap:6px;margin:12px 0;font-weight:650}input,select,textarea,button{font:inherit}
input,select,textarea{padding:10px;border:1px solid #bdcbd3;border-radius:9px;background:white}
textarea{min-height:86px}button{padding:10px 14px;border:1px solid #b8c7cf;border-radius:10px;background:white;cursor:pointer}
button.primary{background:#164e63;color:white;border-color:#164e63;font-weight:800}
.notice{padding:12px 14px;background:#eef7f8;border-left:4px solid #0f766e;border-radius:8px;margin:14px 0}
.warning{padding:10px 12px;background:#fff7ed;border-left:4px solid #c2410c;border-radius:8px}
.steps{display:grid;grid-template-columns:repeat(6,1fr);gap:6px}
.steps span{text-align:center;padding:8px;background:#e7edf1;border-radius:8px;font-size:12px}
.steps .active{background:#164e63;color:white;font-weight:800}.steps .done{background:#dff4ef;color:#115e59}
.ai{border:2px solid #3b82a0;background:#f2fbfd;border-radius:14px;padding:18px;margin:16px 0}
.ai .big{font-size:30px;font-weight:850;margin:8px 0}
.choices{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:8px;margin:14px 0}
.choices button.selected{background:#e8f5f7;border-color:#164e63;font-weight:800}
.preview{display:block;max-width:560px;width:100%;margin:14px auto;border-radius:10px;background:#081017}
pre{white-space:pre-wrap;overflow-wrap:anywhere;background:#0f1720;color:#d7edf2;padding:12px;border-radius:10px;max-height:300px;overflow:auto}
.actions{display:flex;gap:10px;flex-wrap:wrap}.badge{padding:6px 10px;border-radius:999px;background:#dff4ef;color:#115e59;font-weight:800;font-size:12px}
footer{text-align:center;color:#6a7b85;font-size:12px;margin-top:24px}
@media(max-width:760px){header{display:block}.steps{grid-template-columns:repeat(3,1fr)}}
</style>
</head>
<body>
<main class="shell">
<header>
<div><div class="eyebrow">Formative research study</div><h1>NeuroFHIR-Review</h1><div class="muted">Longitudinal neuroimaging evidence review</div></div>
<div class="badge">Research prototype</div>
</header>
<div id="app" class="card">Loading…</div>
<footer>Public de-identified MRI + standardized synthetic workflow context • No patient-care use</footer>
</main>

<script>
let D=null;
const SCREENS=["Case brief","Evidence review","Initial judgment","AI review","Evidence Passport","Final action"];
let S={
 started:false,participant:"",tier:"domain-adjacent",sequence:"",
 order:[],index:0,screen:0,events:[],responses:{},
 initial:"",initialConf:3,finalAction:"",finalConf:3,reason:"",rationale:"",
 provenanceOpened:false,session:""
};
const q=s=>document.querySelector(s);
function now(){return new Date().toISOString()}
function esc(v){return String(v??"").replace(/[&<>"']/g,m=>({"&":"&amp;","<":"&lt;",">":"&gt;","\"":"&quot;","'":"&#039;"}[m]))}
function hashText(t){let h=2166136261;for(let i=0;i<t.length;i++){h^=t.charCodeAt(i);h=Math.imul(h,16777619)}return h>>>0}
function assignSequence(id){return (hashText("sequence-"+id)%2===0)?"A":"B"}
function conditionFor(caseIndex,sequence){
 const odd=(caseIndex%2===0);
 if(sequence==="A") return odd?"evidence-first":"ai-first";
 return odd?"ai-first":"evidence-first";
}
function shuffle(items,seed){let out=[...items],x=hashText(seed);function r(){x=(Math.imul(x,1664525)+1013904223)>>>0;return x/4294967296}for(let i=out.length-1;i>0;i--){const j=Math.floor(r()*(i+1));[out[i],out[j]]=[out[j],out[i]]}return out}
function current(){return S.order[S.index]}
function originalScenarioIndex(c){return Number(c.scenario_id.slice(1))-1}
function condition(){return conditionFor(originalScenarioIndex(current()),S.sequence)}
function path(){return S.tier==="neuro-specialist"?"A":"B"}
function log(type,extra={}){
 if(!S.started||!current())return;
 S.events.push({
  session_id:S.session,participant_id:S.participant,reviewer_tier:S.tier,
  study_path:path(),sequence:S.sequence,scenario_id:current().scenario_id,
  condition:condition(),event_type:type,event_utc:now(),screen:SCREENS[S.screen],...extra
 });
 localStorage.setItem("neurofhir-review-nb14-events",JSON.stringify(S.events));
}
function download(name,text,type="application/json"){
 const blob=new Blob([text],{type}),url=URL.createObjectURL(blob),a=document.createElement("a");
 a.href=url;a.download=name;a.click();URL.revokeObjectURL(url)
}
function csvEsc(v){return '"'+String(v??"").replaceAll('"','""')+'"'}
function resetFields(){S.initial="";S.initialConf=3;S.finalAction="";S.finalConf=3;S.reason="";S.rationale="";S.provenanceOpened=false}
function startStudy(){
 const id=q("#participant").value.trim();
 if(!id){alert("Enter a pseudonymous participant ID.");return}
 S.participant=id;S.tier=q("#tier").value;S.sequence=assignSequence(id);
 S.order=shuffle(D.cases,"order-"+id);S.started=true;S.session="NFR-"+id+"-"+Date.now();
 S.index=0;S.screen=0;S.events=[];S.responses={};resetFields();render();log("case_opened")
}
function nextScreen(n){
 S.screen=n;render();log("screen_opened");
 if(n===1)log("evidence_opened");
 if(n===3)log("ai_exposed",{ai_visible_before_initial_judgment:condition()==="ai-first"});
 if(n===4)log("passport_opened");
}
function chooseInitial(v){S.initial=v;render()}
function chooseFinal(v){S.finalAction=v;render()}
function submitInitial(){
 if(!S.initial){alert("Choose an initial judgment.");return}
 S.initialConf=Number(q("#initialConf").value);
 S.responses[current().scenario_id]={
  scenario_id:current().scenario_id,case_id:current().case_id,
  initial_judgment:S.initial,initial_confidence:S.initialConf,
  initial_submitted_utc:now()
 };
 log("initial_judgment_submitted",{initial_judgment:S.initial,initial_confidence:S.initialConf,ai_visible_before_initial_judgment:condition()==="ai-first"});
 nextScreen(3)
}
function toggleProv(){
 S.provenanceOpened=!S.provenanceOpened;
 if(S.provenanceOpened)log("provenance_opened",{provenance_opened:true});
 render()
}
function submitFinal(){
 S.finalConf=Number(q("#finalConf").value);S.reason=q("#reason").value;S.rationale=q("#rationale").value.slice(0,400);
 if(!S.finalAction||!S.reason){alert("Choose a final action and reason code.");return}
 S.responses[current().scenario_id]={
  ...(S.responses[current().scenario_id]||{}),
  final_action:S.finalAction,final_confidence:S.finalConf,reason_code:S.reason,
  rationale:S.rationale,provenance_opened:S.provenanceOpened,final_submitted_utc:now()
 };
 log("final_action_submitted",{final_action:S.finalAction,final_confidence:S.finalConf,reason_code:S.reason,rationale:S.rationale,provenance_opened:S.provenanceOpened});
 S.index++;S.screen=0;resetFields();
 if(S.index<D.cases.length){render();log("case_opened")}else render()
}
function exportJSON(){
 download(`neurofhir_review_${S.participant}.json`,JSON.stringify({
  exported_utc:now(),participant_id:S.participant,reviewer_tier:S.tier,
  study_path:path(),sequence:S.sequence,responses:S.responses,events:S.events
 },null,2))
}
function exportCSV(){
 const cols=["participant_id","reviewer_tier","study_path","sequence","scenario_id","case_id","condition","initial_judgment","initial_confidence","provenance_opened","final_action","final_confidence","reason_code","rationale","initial_submitted_utc","final_submitted_utc"];
 const rows=Object.values(S.responses).map(r=>({
  participant_id:S.participant,reviewer_tier:S.tier,study_path:path(),sequence:S.sequence,
  condition:conditionFor(Number(r.scenario_id.slice(1))-1,S.sequence),...r
 }));
 const text=[cols.map(csvEsc).join(","),...rows.map(r=>cols.map(c=>csvEsc(r[c])).join(","))].join("\n");
 download(`neurofhir_review_${S.participant}.csv`,text,"text/csv")
}
function steps(){return `<div class="steps">${SCREENS.map((x,i)=>`<span class="${i===S.screen?"active":i<S.screen?"done":""}">${i+1}. ${esc(x)}</span>`).join("")}</div>`}
function aiBox(c){return `<div class="ai"><b>AI recommendation</b><div class="big">${esc(c.ai.conclusion)}</div><div class="grid"><div class="metric"><span>Confidence</span><b>${esc(c.ai.confidence_label)}</b></div><div class="metric"><span>Displayed QC state</span><b>${esc(c.ai.displayed_qc_state)}</b></div></div><div class="warning">${esc(c.ai.known_limitation)}</div></div>`}
function render(){
 const app=q("#app");
 if(!S.started){
  app.innerHTML=`<h2>Study setup</h2><p class="muted">Use the pseudonymous participant ID provided by the study coordinator. Do not enter PHI.</p><div class="grid"><label>Participant ID<input id="participant" placeholder="P001"></label><label>Reviewer category<select id="tier"><option value="neuro-specialist">Neuro specialist</option><option value="domain-adjacent" selected>Clinical/biomedical/imaging-AI researcher</option><option value="trained-reviewer">Trained reviewer</option></select></label></div><div class="notice">${esc(D.study_boundary)}</div><button class="primary" onclick="startStudy()">Start review</button>`;
  return
 }
 if(S.index>=D.cases.length){
  app.innerHTML=`<h2>Review complete</h2><p>Export both study files and return them using the study coordinator's approved process.</p><div class="actions"><button class="primary" onclick="exportCSV()">Export case responses CSV</button><button onclick="exportJSON()">Export interaction log JSON</button></div>`;
  return
 }
 const c=current(),cond=condition(),aiEarly=cond==="ai-first",p=path();
 let body=`<div style="display:flex;justify-content:space-between;gap:12px"><div><div class="eyebrow">Case review</div><h2>${esc(c.display_title)}</h2></div><div class="badge">${S.index+1} / ${D.cases.length}</div></div>${steps()}`;
 if(S.screen===0){
  body+=`<h3>1. Case brief</h3><p class="muted">Review the standardized case evidence. The study interface may present information in different sequences across cases.</p><div class="metric"><span>Case ID</span><b>${esc(c.case_id)}</b></div><button class="primary" onclick="nextScreen(1)">Review evidence</button>`
 }
 if(S.screen===1){
  body+=`<h3>2. Evidence review</h3><img class="preview" src="${esc(c.preview_file)}" alt="Neuroimaging evidence preview"><div class="grid"><div class="metric"><span>Prior volume</span><b>${esc(c.longitudinal.prior_volume_ml)} mL</b></div><div class="metric"><span>Current AI-derived volume</span><b>${esc(c.longitudinal.current_volume_ml)} mL</b></div><div class="metric"><span>Change</span><b>${esc(c.longitudinal.percent_change)}%</b></div></div><div class="notice">${esc(c.longitudinal.context_note)}</div>${aiEarly?aiBox(c):""}<button class="primary" onclick="nextScreen(2)">Record first judgment</button>`
 }
 if(S.screen===2){
  const opts=p==="A"?D.path_A_initial_labels:D.path_B_initial_labels;
  body+=`<h3>3. Initial judgment</h3><p class="muted">${aiEarly?"Base your first judgment on all information currently visible.":"Base your first judgment on the evidence shown so far."}</p><div class="choices">${opts.map(o=>`<button class="${S.initial===o?"selected":""}" onclick='chooseInitial(${JSON.stringify(o)})'>${esc(o)}</button>`).join("")}</div><label>Confidence: <b>${S.initialConf}/5</b><input id="initialConf" type="range" min="1" max="5" value="${S.initialConf}" oninput="S.initialConf=Number(this.value);render()"></label><button class="primary" onclick="submitInitial()">Submit first judgment</button>`
 }
 if(S.screen===3){
  body+=`<h3>4. AI review</h3>${aiBox(c)}<button class="primary" onclick="nextScreen(4)">Inspect Evidence Passport</button>`
 }
 if(S.screen===4){
  body+=`<h3>5. AI Evidence Passport</h3><div class="grid"><div class="metric"><span>Model</span><b>${esc(c.passport.model_name)}</b></div><div class="metric"><span>Version</span><b>${esc(c.passport.model_version)}</b></div><div class="metric"><span>Input QC</span><b>${esc(c.passport.input_qc)}</b></div><div class="metric"><span>Provenance</span><b>${esc(c.passport.provenance_state)}</b></div></div><div class="notice">${esc(c.passport.processing_context)}</div><button onclick="toggleProv()">${S.provenanceOpened?"Hide provenance details":"Open provenance details"}</button>${S.provenanceOpened?`<pre>${esc(JSON.stringify({provenance_state:c.passport.provenance_state,note:c.passport.provenance_note,intended_use:c.passport.intended_use},null,2))}</pre>`:""}<p><button class="primary" onclick="nextScreen(5)">Make final decision</button></p>`
 }
 if(S.screen===5){
  body+=`<h3>6. Final action</h3><div class="choices">${D.final_actions.map(o=>`<button class="${S.finalAction===o?"selected":""}" onclick='chooseFinal(${JSON.stringify(o)})'>${esc(o)}</button>`).join("")}</div><label>Final confidence: <b>${S.finalConf}/5</b><input id="finalConf" type="range" min="1" max="5" value="${S.finalConf}" oninput="S.finalConf=Number(this.value);render()"></label><label>Reason code<select id="reason"><option value="">Choose…</option>${D.reason_codes.map(r=>`<option value="${esc(r)}" ${S.reason===r?"selected":""}>${esc(r)}</option>`).join("")}</select></label><label>Short rationale (optional)<textarea id="rationale" maxlength="400">${esc(S.rationale)}</textarea></label><div class="notice">${p==="A"?"This is a formative expert-review study, not clinical deployment validation.":"This is a workflow-review task, not a diagnostic-accuracy test."}</div><button class="primary" onclick="submitFinal()">Submit and continue</button>`
 }
 app.innerHTML=body
}
fetch("participant_cases.json").then(r=>r.json()).then(x=>{D=x;render()}).catch(e=>{q("#app").innerHTML="<b>Failed to load study data:</b> "+esc(e)})
</script>
</body>
</html>
"""

APP_HTML.write_text(html, encoding="utf-8")

print("=" * 108)
print("✅ Hardened participant app generated")
print("✅ Sequence assignment is automatic from participant ID")
print("✅ Sequence/condition labels are not shown in the participant UI")
print("✅ Neutral case titles are used")
print("✅ Ground-truth/reference answers are stored only in researcher manifest")
print(f"🌐 {APP_HTML}")
print("=" * 108)


✅ Hardened participant app generated
✅ Sequence assignment is automatic from participant ID
✅ Sequence/condition labels are not shown in the participant UI
✅ Neutral case titles are used
✅ Ground-truth/reference answers are stored only in researcher manifest
🌐 /content/drive/MyDrive/neurofhir-qc/wish_extension/notebook_14_distinct_case_study/participant_app/index.html


In [7]:
# Cell 7 — Protocol-hardening validation: uniqueness, balance, blinding, and leakage checks

# 1) Underlying MRI uniqueness.
assert len(researcher_cases) == 12
assert len({c["source_case_id"] for c in researcher_cases}) == 12
assert len({c["participant_case_id"] for c in researcher_cases}) == 12

# 2) Original demo cases excluded.
assert not ({c["source_case_id"] for c in researcher_cases} & original_source_cases)

# 3) Each sequence has 6 Evidence-First + 6 AI-First.
balance = {}
for sequence in ("A", "B"):
    counts = {"evidence-first": 0, "ai-first": 0}
    for c in researcher_cases:
        counts[c["condition_by_sequence"][sequence]] += 1
    balance[sequence] = counts
    assert counts == {"evidence-first": 6, "ai-first": 6}

# 4) Each scenario reverses condition across the two sequences.
for c in researcher_cases:
    assert c["condition_by_sequence"]["A"] != c["condition_by_sequence"]["B"]

# 5) Participant-safe JSON must not contain researcher-only answer/performance fields.
participant_text = PARTICIPANT_MANIFEST.read_text(encoding="utf-8").lower()
forbidden_terms = [
    "source_case_id",
    "case_type",
    "ai_correctness",
    "reference_workflow_disposition",
    "whole_tumor_dice",
    "source_reference_volume_ml",
    "pending-legitimate-expert-reference",
    "wrong-but-plausible",
    "longitudinal-discordance",
]
leaks = [term for term in forbidden_terms if term in participant_text]
if leaks:
    raise AssertionError(f"Participant manifest leaks researcher-only fields: {leaks}")

# 6) Participant UI must not visibly label the experimental condition or sequence.
html_text = APP_HTML.read_text(encoding="utf-8")
visible_forbidden = [
    ">Evidence-First<",
    ">AI-First<",
    "Counterbalance sequence",
    "wrong-but-plausible",
    "expected disposition",
]
visible_leaks = [term for term in visible_forbidden if term in html_text]
if visible_leaks:
    raise AssertionError(f"Participant UI contains avoidable experimental leakage: {visible_leaks}")

# 7) Participant manifest must contain exactly 12 neutral review cases.
participant_payload = load_json(PARTICIPANT_MANIFEST)
assert participant_payload["case_count"] == 12
assert len(participant_payload["cases"]) == 12
assert all(c["display_title"].startswith("Review Case ") for c in participant_payload["cases"])

# 8) Simulate many pseudonymous IDs to confirm hidden deterministic assignment is approximately balanced.
def sequence_for_id(participant_id: str) -> str:
    # Python equivalent of the app's FNV-1a 32-bit hash.
    h = 2166136261
    for ch in ("sequence-" + participant_id):
        h ^= ord(ch)
        h = (h * 16777619) & 0xFFFFFFFF
    return "A" if h % 2 == 0 else "B"

simulated = [sequence_for_id(f"P{i:03d}") for i in range(1, 101)]
sim_counts = {"A": simulated.count("A"), "B": simulated.count("B")}

HARDENING_REPORT = EVAL_ROOT / "protocol_hardening_report.json"
write_json(
    HARDENING_REPORT,
    {
        "generated_utc": utc_now(),
        "distinct_underlying_mri_cases": 12,
        "duplicate_source_case_count": 0,
        "original_demo_case_overlap": 0,
        "condition_balance": balance,
        "participant_manifest_leak_count": len(leaks),
        "participant_ui_visible_leak_count": len(visible_leaks),
        "simulated_100_participant_sequence_assignment": sim_counts,
        "passed": True,
    },
)

print("=" * 108)
print("✅ PROTOCOL HARDENING PASSED")
print("✅ 12/12 unique public MRI source cases")
print("✅ No original demonstration source case reused")
print(f"✅ Sequence A balance: {balance['A']}")
print(f"✅ Sequence B balance: {balance['B']}")
print("✅ Research-answer leakage to participant manifest: 0")
print("✅ Avoidable visible condition/sequence leakage in UI: 0")
print(f"✅ Hidden sequence assignment check across 100 IDs: {sim_counts}")
print(f"📄 {HARDENING_REPORT}")
print("=" * 108)


✅ PROTOCOL HARDENING PASSED
✅ 12/12 unique public MRI source cases
✅ No original demonstration source case reused
✅ Sequence A balance: {'evidence-first': 6, 'ai-first': 6}
✅ Sequence B balance: {'evidence-first': 6, 'ai-first': 6}
✅ Research-answer leakage to participant manifest: 0
✅ Avoidable visible condition/sequence leakage in UI: 0
✅ Hidden sequence assignment check across 100 IDs: {'A': 51, 'B': 49}
📄 /content/drive/MyDrive/neurofhir-qc/wish_extension/notebook_14_distinct_case_study/evaluation/protocol_hardening_report.json


In [8]:
# Cell 8 — Create study-coordinator packet, final audit, and handoff to Notebook 15

COORDINATOR_MD = DOC_ROOT / "STUDY_COORDINATOR_README.md"
AUDIT_JSON = EVAL_ROOT / "notebook_14_distinct_case_protocol_hardening_audit.json"
AUDIT_MD = DOC_ROOT / "NOTEBOOK_14_FINAL_AUDIT.md"

coordinator_text = f"""
# NeuroFHIR-Review — Pilot Study Coordinator Guide

## Use this app
`{PARTICIPANT_ROOT / 'index.html'}`

Serve the `participant_app/` folder from a static web server. Do not give participants
the `researcher_only/` folder.

## Participant ID
Assign a pseudonymous ID such as `P001`. The app deterministically assigns the
counterbalance sequence from that ID. The participant does not choose or see the sequence.

## Review categories
- Neuro specialist -> Path A
- Clinical/biomedical/imaging-AI researcher -> Path B1
- Trained reviewer -> Path B2

## Data collection
At session completion, collect:
1. `neurofhir_review_<ID>.csv`
2. `neurofhir_review_<ID>.json`

Keep the mapping between real identity and pseudonymous ID outside the study app,
using the institutionally approved process.

## Important study boundary
The MRI is public de-identified research data. The longitudinal prior measurement,
dates/context, displayed AI confidence, QC state, and provenance completeness are
standardized/synthetic study manipulations. They do not represent true longitudinal
history of the public-image donor.

## Path A
Do not calculate or claim clinical correctness until a legitimate expert/reference
procedure is established. Current researcher manifest records Path A reference status
as pending.

## Path B
The predefined reference outcome is workflow disposition:
accept / flag / escalate according to the standardized scenario rule.
It is not medical diagnostic accuracy.

## Before recruitment
Complete the IRB/ethics/exemption process required by the institution conducting the study.
Run at least one internal dry run to verify:
- image loading,
- case progression,
- event logging,
- CSV export,
- JSON export,
- no participant-visible research answer.
""".strip()

COORDINATOR_MD.write_text(coordinator_text + "\n", encoding="utf-8")

required_outputs = [
    SELECTION_JSON,
    PREPARED_JSON,
    MODEL_RESULTS_JSON,
    RESEARCH_MANIFEST,
    PARTICIPANT_MANIFEST,
    APP_HTML,
    HARDENING_REPORT,
    COORDINATOR_MD,
]
missing_outputs = [
    str(p) for p in required_outputs if not p.exists() or p.stat().st_size == 0
]
if missing_outputs:
    raise AssertionError(
        "Notebook 14 final gate failed. Missing:\n"
        + "\n".join(f" - {p}" for p in missing_outputs)
    )

audit = {
    "status": "completed",
    "audited_utc": utc_now(),
    "notebook": (
        "14_NeuroFHIR_Review_WISH_Distinct_Case_Expansion_and_Protocol_Hardening.ipynb"
    ),
    "metrics": {
        "distinct_public_mri_cases": 12,
        "original_demo_cases_reused": 0,
        "model_inference_cases": len(model_rows),
        "participant_scenarios": len(participant_cases),
        "sequence_A": balance["A"],
        "sequence_B": balance["B"],
        "participant_manifest_answer_leaks": 0,
        "participant_ui_visible_condition_leaks": 0,
    },
    "scientific_corrections_from_notebook_13": [
        "No participant sees the same underlying MRI case twice.",
        "Counterbalance sequence is assigned automatically rather than participant-selected.",
        "Scenario challenge/type labels are hidden from participants.",
        "Expected workflow disposition is researcher-only.",
        "Model performance/reference-mask metrics are researcher-only.",
        "Participant sees neutral case labels.",
        "Path A reference is explicitly pending legitimate expert/reference procedure.",
        "Path B reference remains workflow disposition, not diagnostic accuracy.",
    ],
    "next_step_before_notebook_15": [
        "Run an internal dry run of the participant app.",
        "Finalize reviewer training material.",
        "Complete applicable IRB/ethics/exemption process.",
        "Recruit Path A expert if feasible; otherwise execute Path B.",
        "Collect real participant CSV + JSON exports.",
    ],
    "next_notebook": (
        "15_NeuroFHIR_Review_WISH_Pilot_Data_Ingestion_and_Human_AI_Analysis.ipynb"
    ),
    "final_gate": True,
}
write_json(AUDIT_JSON, audit)

AUDIT_MD.write_text(
    f"""# Notebook 14 — Final Audit

**Status:** completed

## Study hardening completed
- 12 distinct public MRI cases.
- 0 original demonstration source cases reused.
- Same pinned MONAI model executed on all 12.
- One unique MRI scenario per participant.
- 6 Evidence-First / 6 AI-First per sequence.
- Automatic hidden counterbalancing.
- Research answers separated from participant app.
- Neutral case labels.
- Path A reference status explicitly pending.
- Path B workflow-reference framing preserved.

## Participant app
`{PARTICIPANT_ROOT / 'index.html'}`

## Researcher-only key
`{RESEARCH_MANIFEST}`

## Next notebook
`15_NeuroFHIR_Review_WISH_Pilot_Data_Ingestion_and_Human_AI_Analysis.ipynb`

Do not run the human-AI outcome analysis until real pilot exports exist.
""",
    encoding="utf-8",
)

print("=" * 108)
print("✅ NOTEBOOK 14 FINAL GATE: TRUE")
print("✅ The study now has 12 distinct underlying MRI cases")
print("✅ Participant-facing experimental leakage has been removed")
print("✅ Counterbalancing is hidden and automatic")
print(f"👤 Pilot app: {PARTICIPANT_ROOT / 'index.html'}")
print(f"🔒 Researcher key: {RESEARCH_MANIFEST}")
print("")
print("NEXT: internal dry run → recruitment/data collection → Notebook 15 analysis")
print("=" * 108)


✅ NOTEBOOK 14 FINAL GATE: TRUE
✅ The study now has 12 distinct underlying MRI cases
✅ Participant-facing experimental leakage has been removed
✅ Counterbalancing is hidden and automatic
👤 Pilot app: /content/drive/MyDrive/neurofhir-qc/wish_extension/notebook_14_distinct_case_study/participant_app/index.html
🔒 Researcher key: /content/drive/MyDrive/neurofhir-qc/wish_extension/notebook_14_distinct_case_study/researcher_only/researcher_scenario_key.json

NEXT: internal dry run → recruitment/data collection → Notebook 15 analysis
